# 01 — Understand and validate the data

## Goal

Before modelling, we need to understand what one row represents and whether the supplied data are safe to use.

In this notebook we will answer:

1. How many rows and columns are in each file?
2. What is the prediction target?
3. Are IDs unique and are there duplicated rows?
4. Which values are missing?
5. Are categorical values and numeric ranges plausible?
6. Do the training and unknown datasets have compatible columns and distributions?

**Important:** This is a data-quality notebook. We will not train a model yet.

## Setup

The paths below are relative to the project, so another person can rerun the notebook after placing the two CSV files in `data/raw/`.

In [ ]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 50)

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / "data" / "raw"
TRAIN_PATH = DATA_DIR / "Assignment3-TrainingDataset.csv"
UNKNOWN_PATH = DATA_DIR / "Assignment3-UnknownDataset.csv"

assert TRAIN_PATH.exists(), f"Training file not found: {TRAIN_PATH}"
assert UNKNOWN_PATH.exists(), f"Unknown file not found: {UNKNOWN_PATH}"

print(f"Project directory: {PROJECT_DIR}")

## Steps

### 1. Load the data

The training file includes the answer (`satisfaction`). The unknown file contains the passengers for whom a prediction would eventually be produced.

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
unknown_df = pd.read_csv(UNKNOWN_PATH)

print(f"Training shape: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns")
print(f"Unknown shape:  {unknown_df.shape[0]:,} rows × {unknown_df.shape[1]} columns")

display(train_df.head(3))

### 2. Confirm the modelling grain and target

Our working assumption is that **one row represents one passenger survey for one flight experience**. The `id` should therefore be unique within and across the two files.

The target is binary:

- `satisfied`
- `neutral or dissatisfied`

Combining neutral and dissatisfied passengers into one class is a dataset design decision. It simplifies modelling, but it also removes the distinction between neutral and genuinely unhappy passengers.

In [ ]:
target_counts = train_df["satisfaction"].value_counts().rename("count").to_frame()
target_counts["percentage"] = (target_counts["count"] / len(train_df) * 100).round(2)

display(target_counts)

print(f"Unique training IDs: {train_df['id'].nunique():,} of {len(train_df):,}")
print(f"Unique unknown IDs:  {unknown_df['id'].nunique():,} of {len(unknown_df):,}")
print(f"IDs appearing in both files: {len(set(train_df['id']) & set(unknown_df['id'])):,}")

In [ ]:
target_display = target_counts.copy()
target_display["share_bar"] = target_display["percentage"].apply(
    lambda percentage: "█" * round(percentage / 2)
)
display(target_display)

**How to interpret this:** A 56.67% versus 43.33% split is only mildly uneven. Accuracy is still useful as a descriptive metric, but it must be compared with a simple majority-class baseline and supplemented with precision, recall, F1 and ranking metrics.

### 3. Check columns and data types

The unknown dataset should contain the same predictor columns as training, with only the target omitted.

In [ ]:
training_features = [column for column in train_df.columns if column != "satisfaction"]

print("Columns only in training:", sorted(set(train_df.columns) - set(unknown_df.columns)))
print("Columns only in unknown:", sorted(set(unknown_df.columns) - set(training_features)))

column_profile = pd.DataFrame({
    "training_dtype": train_df.dtypes.astype(str),
    "training_unique": train_df.nunique(dropna=False),
    "training_missing": train_df.isna().sum(),
})
display(column_profile)

### 4. Check missing values and duplicates

Missingness should be measured as a rate, not only a count. We also distinguish repeated IDs from completely duplicated rows.

In [ ]:
def missing_value_profile(dataframe: pd.DataFrame) -> pd.DataFrame:
    profile = pd.DataFrame({
        "missing_count": dataframe.isna().sum(),
        "missing_percentage": dataframe.isna().mean().mul(100).round(3),
    })
    return profile.loc[profile["missing_count"] > 0].sort_values("missing_percentage", ascending=False)

print("Training missing values")
display(missing_value_profile(train_df))

print("Unknown missing values")
display(missing_value_profile(unknown_df))

print(f"Exact duplicated training rows: {train_df.duplicated().sum():,}")
print(f"Exact duplicated unknown rows:  {unknown_df.duplicated().sum():,}")

### 5. Check categorical values

Unexpected spelling, capitalization or whitespace can create false categories during one-hot encoding.

In [ ]:
categorical_columns = train_df.select_dtypes(include="object").columns.tolist()

category_checks = []
for column in categorical_columns:
    category_checks.append({
        "column": column,
        "training_values": sorted(train_df[column].dropna().astype(str).unique().tolist()),
        "unknown_values": (
            sorted(unknown_df[column].dropna().astype(str).unique().tolist())
            if column in unknown_df.columns
            else "Target column"
        ),
    })

display(pd.DataFrame(category_checks))

### 6. Check numeric ranges

The service questions are described as rating scales, but several contain zero. We should not automatically assume zero means a genuine rating—it may represent “not applicable” or “not answered.” Because no data dictionary was supplied, this remains an open question and must be documented rather than guessed.

In [ ]:
numeric_columns = [
    column for column in training_features
    if pd.api.types.is_numeric_dtype(train_df[column]) and column != "id"
]

numeric_profile = pd.DataFrame({
    "train_min": train_df[numeric_columns].min(),
    "train_median": train_df[numeric_columns].median(),
    "train_max": train_df[numeric_columns].max(),
    "train_zero_pct": train_df[numeric_columns].eq(0).mean().mul(100).round(2),
    "unknown_min": unknown_df[numeric_columns].min(),
    "unknown_median": unknown_df[numeric_columns].median(),
    "unknown_max": unknown_df[numeric_columns].max(),
    "unknown_zero_pct": unknown_df[numeric_columns].eq(0).mean().mul(100).round(2),
})

display(numeric_profile)

### 7. Compare broad train/unknown distributions

The unknown set does not need to be identical to training, but large shifts can make model performance less reliable. This first pass compares numeric medians and categorical proportions.

In [ ]:
distribution_comparison = pd.DataFrame({
    "train_median": train_df[numeric_columns].median(),
    "unknown_median": unknown_df[numeric_columns].median(),
})
distribution_comparison["absolute_difference"] = (
    distribution_comparison["unknown_median"] - distribution_comparison["train_median"]
).abs()

display(distribution_comparison.sort_values("absolute_difference", ascending=False))

In [ ]:
for column in ["Gender", "Customer Type", "Type of Travel", "Class"]:
    comparison = pd.concat(
        [
            train_df[column].value_counts(normalize=True).rename("training_share"),
            unknown_df[column].value_counts(normalize=True).rename("unknown_share"),
        ],
        axis=1,
    ).fillna(0)
    comparison["percentage_point_gap"] = (
        comparison["unknown_share"] - comparison["training_share"]
    ).mul(100).round(2)
    print(f"\n{column}")
    comparison[["training_share", "unknown_share"]] = comparison[
        ["training_share", "unknown_share"]
    ].mul(100).round(2)
    display(comparison.rename(columns={
        "training_share": "training_percentage",
        "unknown_share": "unknown_percentage",
    }))

## Checks

The assertions below turn our most important expectations into executable tests. If a future version of the files violates one of them, the notebook stops instead of silently continuing.

In [ ]:
assert train_df["id"].is_unique, "Training IDs are not unique."
assert unknown_df["id"].is_unique, "Unknown IDs are not unique."
assert not (set(train_df["id"]) & set(unknown_df["id"])), "The files contain overlapping IDs."
assert set(train_df["satisfaction"].dropna().unique()) == {
    "satisfied",
    "neutral or dissatisfied",
}, "Unexpected target labels found."
assert training_features == unknown_df.columns.tolist(), "Predictor columns differ or are out of order."

print("All structural checks passed.")

## Next Steps

### Verified findings

- Training contains **83,123 passengers** and unknown contains **20,781 passengers**.
- The target is **43.33% satisfied** and **56.67% neutral or dissatisfied**.
- IDs are unique within each file and do not overlap across files.
- There are no exact duplicated rows.
- `Arrival Delay in Minutes` is the only column with null values: **249 training rows (0.300%)** and **61 unknown rows (0.294%)**.
- Training and unknown contain compatible predictor columns.

### Open questions

- What do zero values in the service-rating fields mean? A data dictionary is needed before recoding them.
- What is the original source and redistribution licence of the data?
- Is the intended use post-flight explanation or prediction before/during a flight? This determines whether arrival-delay information is valid at prediction time.
- Should “neutral” and “dissatisfied” really be combined for the business decision?

### Next notebook

The next stage will perform exploratory analysis. We will investigate which passenger groups and service attributes are associated with satisfaction, while carefully avoiding causal language.